EşdeğerDers - Frontend (Kişi 1) Entegrasyon Rehberi

Bu notebook, arayüzden backend'e (/api/esdeger-ders/generate) gönderilecek veriyi, dönen alanların yapısını ve bileşenlere nasıl dağıtılacağını gösterir.

In [ ]:
import requests
import json

# Backend adresleri
API_URL = "http://127.0.0.1:8000/api/esdeger-ders/generate"
HEALTH_URL = "http://127.0.0.1:8000/health"

# Sunucu ayakta mı kontrolü
res = requests.get(HEALTH_URL)
print("Sunucu Durumu:", res.status_code, res.json())

1. Frontend Formunun Göndereceği Veri (Payload)

Arayüzdeki formdan alınacak alanlar bu formatta JSON olarak POST edilmelidir:

In [ ]:
# Arayüzden backend'e gidecek örnek form verisi
payload = {
    "kazanim": "Dört eş parçadan üçünü 3/4 olarak temsil eder.",
    "yas_seviye": "9 yaş / 3. Sınıf",
    "malzemeler": [
        "1 adet kâğıt",
        "1 kurşun kalem",
        "8 adet aynı boy taş"
    ],
    "sure_dakika": 30,
    "erisim_kosullari": "İnternet yok, ekran yok, yazıcı yok, makas yok.",
    "prompt_version": "v2"
}

# İsteği atalım
response = requests.post(API_URL, json=payload)
data = response.json()
print("HTTP Kodu:", response.status_code)

2. Arayüz Bileşenleri ile Alanların Eşleşmesi (Mapping)

Gelen JSON yanıtındaki verileri ekrandaki 3 ana alana dağıtabilirsiniz:   
Alert Banner: data["uygulanabilirlik_notu"] 
Etkinlik Kartı: data["etkinlikler"]
Değerlendirme Bölümü: data["degerlendirme"]

In [ ]:
# 1. En üstteki sarı/uyarı öğretmen banner'ı
print("=== ÖĞRETMEN UYARI BANNERI ===")
print("Uyarı:", data["uygulanabilirlik_notu"])
print("\n" + "="*50 + "\n")

# 2. Üç etkinlik kartı
print("=== 3 ETKİNLİK KARTI ===")
for i, act in enumerate(data["etkinlikler"], 1):
    print(f"Kart {i}: {act['ad']} ({act['yontem']}) - {act['tahmini_sure_dakika']} dk")
    print(f"Kullanılan Malzemeler: {', '.join(act['kullanilan_malzemeler']) if act['kullanilan_malzemeler'] else 'Ek malzeme yok'}")
    print("Uygulama Adımları:")
    for step in act["uygulama_adimlari"]:
        print(f"  • {step}")
    print(f"Öğrenci Davranışı: {act['beklenen_ogrenci_davranisi']}\n")

print("="*50 + "\n")

# 3. Ortak değerlendirme sorusu ve cevap anahtarı
print("=== ORTAK DEĞERLENDİRME ===")
print("Soru  :", data["degerlendirme"]["ortak_soru"])
print("Cevap :", data["degerlendirme"]["dogru_cevap"])
print("Ölçüt :", data["degerlendirme"]["degerlendirme_olcutu_ve_gerekce"])

3. Frontend İçin JavaScript fetch Kodu
React, Vue veya düz JS tarafında doğrudan kullanmak istersen:

In [ ]:
const response = await fetch("http://127.0.0.1:8000/api/esdeger-ders/generate", {
  method: "POST",
  headers: { "Content-Type": "application/json" },
  body: JSON.stringify(formData)
});
const result = await response.json();